# Parsing Milestone 2 trên ổ local Colab

Mỗi thành viên chọn TV của mình, đường dẫn snapshot MS1 sau phục hồi và chế độ SMOKE hoặc FULL. Chọn CPU trong Runtime.

Notebook dùng 8 luồng sao chép và lô 50 ID, parsing và QA trên ổ local Colab. Sau mỗi lô, checkpoint được đóng thành ZIP có checksum và lưu lên Drive. Cuối lượt chạy, notebook tự xuất ZIP bàn giao.

Chạy mới: tự tạo index hoàn tất. Chạy tiếp cùng snapshot và runtime: đọc index gọn, bỏ qua source của bài đã qua validation và chỉ chép bài mới hoặc chưa hoàn tất. Paper còn lỗi được ghi trong acceptance, không được tính là đã hoàn tất.

Bản sửa ngày 19/11 dùng thư mục kết quả riêng `ms2-repaired-20261119`. Giữ kết quả parser cũ để đối chiếu; không đổi tag mới về tag cũ để ép dùng cache. Các lần resume sau của bản này giữ nguyên tag mới, thành viên và snapshot.

Giữ snapshot MS1 cố định suốt lượt chạy. Chạy mới: thử SMOKE trước khi chuyển sang FULL. Nếu đang dùng backend 949e5ec và còn checkpoint local: dừng cell, giữ runtime, cập nhật CODE_REF theo bản này rồi chạy lại cell cài đặt, cell xác định phạm vi và bước parsing. Giữ MEMBER, đường dẫn input/output, RUN_TAG và LOCAL_ROOT hiện có.

Notebook dùng backend `e8e9d14`, bổ sung validation BibTeX theo cụm hoàn chỉnh và hiển thị paper đang xử lý. Giữ nguyên tag kết quả, thành viên và snapshot để tiếp tục kết quả tốt của bản sửa ngày 19/11; chứng nhận hoàn tất từ 949e5ec vẫn tương thích. Lô dở dùng lại stage còn hợp lệ, tính lại validation theo bản sửa.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Chọn thành viên và dữ liệu

Chọn `MEMBER` từ TV1 đến TV5, rồi nhập `MS1_INPUT` là thư mục data MS1 của mình. Để trống thì dùng `/content/drive/MyDrive/arxiv_ms1_recovery/<member>/data`. Nếu dùng thư mục được chia sẻ, thêm shortcut vào My Drive và nhập đúng đường dẫn nhìn thấy trên Colab.

Cell cấu hình chỉ có ba mục:

| Mục | Cần chọn gì? |
| --- | --- |
| `MEMBER` | Thành viên của mình, ví dụ `tv3`. Notebook tự lấy đúng dải ID. |
| `MS1_INPUT` | Đường dẫn thư mục data MS1 có `corpus`, `raw`, `state`. Để trống để dùng đường dẫn mặc định nêu trên. |
| `RUN_MODE` | Lần đầu chọn `SMOKE` để thử tối đa ba bài; thử qua rồi chọn `FULL` để xử lý đủ 5.000 ID được giao. |

Sau khi chọn, bấm nút ▶ bên trái cell để áp dụng rồi chạy các cell tiếp theo từ trên xuống. Khi đổi từ SMOKE sang FULL, chạy lại cell cấu hình, cell xác nhận phạm vi và cell parsing; không cần mount hoặc cài đặt lại. Các thiết lập còn lại dùng mặc định: lưu kết quả riêng theo thành viên, tách SMOKE/FULL và tự resume nếu bị ngắt.

Nếu cần dùng MS2 trước bản vá: cấu hình nâng cao `LEGACY_MS2_INPUT` nhận đường dẫn mount tới thư mục kết quả thành viên chứa `parsed/`, `state/` hoặc `local-batches/`. Để trống sẽ tìm `<OUTPUT_ROOT>/ms2-members-20261110/full/<MEMBER>` trước. Thư mục shared cần shortcut trong My Drive; không nhập URL hoặc ZIP. Giữ output/tag mới để tiếp tục các paper vừa chạy xong. Lượt đầu chuyển tiếp vẫn cần chép source và chạy resolver/bibliography hiện tại; số stage dùng lại được in riêng ở `migrated_stages`. Cell cài đặt tải đúng backend đã pin. Nếu notebook báo runtime thiếu chức năng, dừng cell rồi chạy lại cell cài đặt và cell xác định phạm vi với pin mới. Giữ runtime nếu còn checkpoint local của lô dở.

In [ ]:
from pathlib import Path
#@title Cấu hình chạy: chỉ cần chọn ba mục
#@markdown Thành viên: chọn đúng TV của mình; range sẽ được lấy tự động.
MEMBER = 'tv1' #@param ["tv1", "tv2", "tv3", "tv4", "tv5"]
#@markdown Dữ liệu: nhập thư mục data MS1; để trống nếu dùng đường dẫn mặc định.
MS1_INPUT = '' #@param {type:"string"}
#@markdown Chế độ: SMOKE = chạy thử ba bài; FULL = chạy toàn bộ dải ID của mình.
RUN_MODE = 'SMOKE' #@param ["SMOKE", "FULL"]
# Thiết lập mặc định; không cần sửa khi chạy theo hướng dẫn.
OUTPUT_ROOT = '/content/drive/MyDrive/arxiv_ms2_results'
RUN_TAG = 'ms2-repaired-20261119'
SMOKE_IDS = ''
# Nguồn MS2 cũ: đường dẫn mount tới thư mục thành viên; trống = tự tìm tag cũ.
LEGACY_MS2_INPUT = ''
VERIFY_RESUME = False
# Cùng mã production đã nghiệm thu; không đổi riêng từng người.
CODE_REF = 'e8e9d1485748d0f480a3dc5aedd27c7b0efb0e7f'
RESUME = True
LOCAL_ROOT = Path('/content/arxiv_ms2_local')


In [ ]:
import importlib, subprocess, sys
REPO = Path('/content/arxiv-ms2-integration')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/LeoTkiet/arxiv-data-science.git', str(REPO)], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Checkout has local changes; keep them before changing CODE_REF.')
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', CODE_REF], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
CODE_SHA = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO) + '[reporting]'], check=True)
# Add the checked-out source even when this runtime previously installed another checkout.
for module_name in list(sys.modules):
    if module_name == 'arxiv_pipeline' or module_name.startswith('arxiv_pipeline.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
if CODE_SHA != CODE_REF:
    raise RuntimeError('Runtime commit differs from the pinned CODE_REF')
print('Commit:', CODE_SHA)

## 2. Xác nhận phạm vi của mình

Notebook đọc phân công từ `configs/ms1_group6_ranges.json` trong cùng checkout. Kiểm tra thành viên, input, output và range in dưới đây trước khi chạy cell parsing.

| Thành viên | Dải ID |
| --- | --- |
| TV1 | 2508.12872–2508.17871 |
| TV2 | 2506.15788–2506.20787 |
| TV3 | 2411.08799–2411.13798 |
| TV4 | 2410.18025–2410.23024 |
| TV5 | 2410.03025–2410.08024 |

In [ ]:
import json, re
from arxiv_pipeline.main import expand_range
from arxiv_pipeline.pipeline import parse

MEMBER_ROWS = {'tv1': 15, 'tv2': 24, 'tv3': 56, 'tv4': 59, 'tv5': 62}
if MEMBER not in MEMBER_ROWS or RUN_MODE not in {'SMOKE', 'FULL'}:
    raise ValueError('Chọn đúng MEMBER và RUN_MODE trong danh sách.')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', RUN_TAG):
    raise ValueError('RUN_TAG chỉ dùng chữ, số, dấu chấm, gạch dưới hoặc gạch nối.')
assignment_config = json.loads((REPO / 'configs/ms1_group6_ranges.json').read_text())
matches = [r for r in assignment_config['ranges'] if r['source_row'] == MEMBER_ROWS[MEMBER]]
if len(matches) != 1:
    raise ValueError('Không xác định được dải phân công từ configs/ms1_group6_ranges.json.')
assignment = matches[0]
assigned_ids = expand_range(assignment['range'])
if len(assigned_ids) != assignment['count']:
    raise ValueError('Dải phân công không khớp số ID được giao.')
input_root = Path(MS1_INPUT.strip() or f'/content/drive/MyDrive/arxiv_ms1_recovery/{MEMBER}/data')
OUTPUT_BASE = Path(OUTPUT_ROOT) / RUN_TAG / RUN_MODE.lower()
parse.validate_roots(input_root, OUTPUT_BASE / MEMBER)
from arxiv_pipeline.pipeline import optimized_staging as runtime_backend
if (not all(callable(getattr(runtime_backend, name, None))
            for name in ('select_smoke_ids', 'validate_legacy'))
        or not getattr(runtime_backend, 'stage_migration', None)):
    raise RuntimeError('runtime hiện tại thiếu backend mới. Mở bản notebook mới trên GitHub, restart session rồi chạy lại cell cài đặt để nạp đúng CODE_REF.')
select_smoke_ids = runtime_backend.select_smoke_ids
validate_legacy = runtime_backend.validate_legacy
job = dict(member=MEMBER, input=str(input_root), ids=None,
           total_assigned=assignment['count'], assignment_range=assignment['range'])
legacy_setting = globals().get('LEGACY_MS2_INPUT', '').strip()
if legacy_setting:
    legacy_root = Path(legacy_setting)
else:
    candidates = [Path(OUTPUT_ROOT) / 'ms2-members-20261110' / mode / MEMBER
                  for mode in dict.fromkeys(['full', RUN_MODE.lower()])]
    legacy_root = next((p for p in candidates if p.is_dir()), None)
if legacy_root is not None:
    legacy_root = validate_legacy(legacy_root, MEMBER)
    job['legacy_output'] = str(legacy_root)
    print('MS2 cũ (chỉ đọc):', legacy_root)
    print('Chỉ tái sử dụng stage có đủ bằng chứng; source MS1 và validation vẫn được kiểm tra.')
else:
    print('Không chọn nguồn MS2 cũ. Chỉ dùng cache của output hiện tại hoặc chạy mới.')
if RUN_MODE == 'FULL':
    selected = None
elif SMOKE_IDS.strip():
    selected = SMOKE_IDS.replace(',', ' ').split()
else:
    selected = select_smoke_ids(job, OUTPUT_BASE / MEMBER, local_root=LOCAL_ROOT)
if selected is not None and (not selected or not set(selected).issubset(assigned_ids)):
    raise ValueError('Cần ID mẫu hợp lệ thuộc dải của thành viên; kiểm tra discovery hoặc SMOKE_IDS.')
job['ids'] = selected
JOBS = [job]
CHECK_RESUME = RUN_MODE == 'SMOKE' or VERIFY_RESUME
print('Thành viên:', MEMBER, '| chế độ:', RUN_MODE, '| range:', assignment['range'])
print('Số ID sẽ xử lý:', len(assigned_ids) if selected is None else len(selected))
print('Input:', input_root)
print('Output:', OUTPUT_BASE / MEMBER)
print('Commit:', CODE_SHA)

## 3. Chạy parsing trên local và lưu checkpoint theo lô

Cell hiển thị số bài hoàn tất được bỏ qua, số ID mới, số ID thử lại và thời gian từng bước: chép input, khôi phục checkpoint, parsing/QA, đóng ZIP và lưu Drive. Lô mặc định có 50 ID; bài mới được ưu tiên trước bài có receipt chưa hoàn tất. Nếu runtime bị ngắt, các lô đã ghi receipt trên Drive được khôi phục ở lượt chạy sau. Lô đang xử lý sẽ được thử lại từ checkpoint đã lưu gần nhất.

Ổ local cần đủ chỗ cho source của một lô và output đang tích lũy của lượt chạy. Khi hết dung lượng hoặc gặp lỗi sao chép, xem thông báo rồi chạy tiếp trên runtime có đủ chỗ. Các gói đã lưu được giữ trong thư mục output trên Drive.

In [ ]:
from arxiv_pipeline.pipeline import optimized_staging
from arxiv_pipeline.pipeline.optimized_staging import run as run_local

if any(job.get('legacy_output') for job in JOBS) and not getattr(optimized_staging, 'stage_migration', None):
    raise RuntimeError('runtime chưa hỗ trợ kiểm tra tương thích stage. Cần bản đã phát hành và pin mới trước khi dùng MS2 cũ.')

# Thiết lập thực thi mặc định; giao diện chọn thành viên vẫn có ba mục.
COPY_WORKERS = 8
BATCH_SIZE = 50
results = []
for job in JOBS:
    destination = OUTPUT_BASE / job['member']
    try:
        result = run_local(
            job, destination, local_root=LOCAL_ROOT, code_sha=CODE_SHA,
            workers=COPY_WORKERS, batch_size=BATCH_SIZE, check_resume=CHECK_RESUME, finalize=True)
    except Exception:
        print('Lượt chạy bị ngắt. Giữ nguyên output Drive để tiếp tục checkpoint:', destination)
        raise
    results.append(result)
    print(job['member'], {
        k: result.get(k, 0) for k in [
            'complete', 'attempted_papers', 'paper_statuses',
            'cached_stages', 'migrated_stages', 'total_stages', 'skipped_completed_papers', 'full_assignment_accepted']})
    print('ZIP bàn giao:', destination / 'MS2_RESULTS.zip')

## 4. Bàn giao kết quả

Sau khi chạy xong, gửi `MS2_RESULTS.zip`, `acceptance.json` và `delivery.json` trong output FULL của mình cho TV1/TV5. ZIP chứa đầy đủ thư mục của thành viên, gồm tables, parsed, state, validation, manifest và các số đo. `delivery.json` ghi checksum ZIP và thời gian xuất file cuối.

Các file trong `local-batches/` phục vụ khôi phục sau gián đoạn. Giữ thư mục này khi tiếp tục lượt chạy. Chỉ tải ZIP kết quả khi cell đã thông báo lưu thành công và delivery có trạng thái published.

Thời gian benchmark gồm đọc discovery, chép input, khôi phục cache, parsing/QA local, lưu checkpoint theo lô và thu thập output cũ để bàn giao. RAM được đo trong bước discovery và xử lý bài còn lại. Thời gian đóng gói/upload ZIP cuối được ghi riêng trong delivery. Số stage dùng cache chỉ tính stage thực thi trong lượt hiện tại; số bài bỏ qua theo index được ghi riêng.

SMOKE và FULL dùng output khác nhau. Để chuyển từ SMOKE sang FULL, đổi RUN_MODE rồi chạy lại cell cấu hình, cell xác nhận phạm vi và cell parsing.

In [ ]:
import json
result_dir = OUTPUT_BASE / MEMBER
receipt = result_dir / 'delivery.json'
if receipt.is_file():
    delivery = json.loads(receipt.read_text())
    print('Trạng thái:', delivery['status'])
    if delivery['status'] == 'published':
        print('ZIP bàn giao:', result_dir / delivery['archive'])
        print('SHA-256:', delivery['sha256'])
        print('Tổng thời gian lượt chạy:', delivery['total_elapsed_seconds'], 'giây')
else:
    print('Chạy cell parsing để tạo kết quả.')